In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

### 读取数据集并查看

In [ ]:
train_data = pd.read_csv(r"DataSet\train.csv")
test_data = pd.read_csv(r"DataSet\test.csv")
train_data

In [ ]:
# 下面的代码是为了填充数据中的缺失值
for col in train_data.columns:
    if col == "SalePrice" or col == "Id":   # 跳过标签列和ID列
        continue
    if pd.api.types.is_numeric_dtype(train_data[col]):  # 判断是否为连续数值型数据
        fill_value = train_data[col].mean()
        # 保持整数类型
        if pd.api.types.is_integer_dtype(train_data[col]):
            fill_value = int(round(fill_value))
        else:
            # 保留原有小数位数
            decimals = train_data[col].apply(lambda x: str(x)[::-1].find('.') if '.' in str(x) else 0).max()
            fill_value = round(float(fill_value), decimals)
        train_data[col].fillna(fill_value, inplace=True)
    else:  # 如果是分类数据
        fill_value = train_data[col].mode()[0]
        train_data[col].fillna(fill_value, inplace=True)

# 对测试数据也进行相同的处理
for col in test_data.columns:
    if col == "Id":   # 跳过ID列
        continue
    if pd.api.types.is_numeric_dtype(test_data[col]):  # 判断是否为连续数值型数据
        fill_value = test_data[col].mean()
        if pd.api.types.is_integer_dtype(test_data[col]):
            fill_value = int(round(fill_value))
        else:
            decimals = test_data[col].apply(lambda x: str(x)[::-1].find('.') if '.' in str(x) else 0).max()
            fill_value = round(float(fill_value), decimals)
        test_data[col].fillna(fill_value, inplace=True)
    else:  # 如果是分类数据
        fill_value = test_data[col].mode()[0]
        test_data[col].fillna(fill_value, inplace=True)

In [ ]:
noise_example = train_data.sample(1).copy()
train_mean = {col: train_data[col].mean() for col in train_data.columns if pd.api.types.is_numeric_dtype(train_data[col])}
train_std = {col: train_data[col].std() for col in train_data.columns if pd.api.types.is_numeric_dtype(train_data[col])}
#用字典存储对应的事先计算的均值与标准差，避免反复计算

for col in noise_example.columns:
    if col == "Id":
        noise_example[col] = train_data[col].max() + 1
    elif pd.api.types.is_numeric_dtype(train_data[col]):
        value = np.random.normal(loc=train_mean[col], scale=train_std[col])
        # 保证生成数据非负，因为一间房子的面积之类的数值不能为负数是吧
        value = np.clip(value, 0, None)
        # 保持格式
        if pd.api.types.is_integer_dtype(train_data[col]):
            value = int(round(value))
        else:
            # 保留与原数据相同的小数位数
            decimals = train_data[col].apply(lambda x: str(x)[::-1].find('.') if '.' in str(x) else 0).max()
            value = round(float(value), decimals)
        noise_example[col] = value
    else:
        noise_example[col] = np.random.choice(train_data[col])

noise_example


In [ ]:
def add_noise_samples(df, n_noise=1, id_col="Id"):
    df = df.copy()
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) and col != id_col]  # 选择数值型列，排除ID列
    categorical_cols = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col]) and col != id_col] # 选择非数值型列，排除ID列
    means = {col: df[col].mean() for col in numeric_cols}
    stds = {col: df[col].std() for col in numeric_cols}
    noise_rows = []
    for _ in range(n_noise):    # 遍历指定数值生成n_noise个噪声样本
        noise = {}
        for col in df.columns:  #生成逻辑与上块代码类似
            if col == id_col:
                noise[col] = df[col].max() + 1 + _
            elif col in numeric_cols:
                value = np.random.normal(loc=means[col], scale=stds[col])
                value = np.clip(value, 0, None)
                if pd.api.types.is_integer_dtype(df[col]):
                    value = int(round(value))
                else:
                    decimals = df[col].apply(lambda x: str(x)[::-1].find('.') if '.' in str(x) else 0).max()
                    value = round(float(value), decimals)
                noise[col] = value
            elif col in categorical_cols:
                noise[col] = np.random.choice(df[col])
            else:
                noise[col] = np.nan
        noise_rows.append(noise)
    noise_df = pd.DataFrame(noise_rows, columns=df.columns) #生成的所有新噪声
    return pd.concat([df, noise_df], ignore_index=True) #将噪声数据与原数据合并后返回

In [ ]:
train_data = add_noise_samples(train_data,20)
train_data.tail()

In [ ]:
train_data = train_data.drop(columns=["Id"])  # 删除ID列
train_data.head()

In [ ]:
numeric_cols = [col for col in train_data.columns if pd.api.types.is_numeric_dtype(train_data[col]) and col != "SalePrice"]  # 排除标签列
train_data[numeric_cols] = train_data[numeric_cols].apply(lambda x: (x - x.mean()) / x.std(), axis=0)  # 正则化数值型数据
train_data.head()

In [ ]:
#对训练集中的离散分类数据进行独热编码
train_data = pd.get_dummies(train_data, dummy_na=True)
train_data.head()

In [22]:
def preprocess_data(df, enhance_noise=False, n_noise=0, id_col="Id", label_col="SalePrice"):
    df = df.copy()
    # 1. 填补缺失值
    for col in df.columns:
        if col == label_col or col == id_col:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            fill_value = df[col].mean()
            if pd.api.types.is_integer_dtype(df[col]):
                fill_value = int(round(fill_value))
            else:
                decimals = df[col].apply(lambda x: str(x)[::-1].find('.') if '.' in str(x) else 0).max()
                fill_value = round(float(fill_value), decimals)
            df[col].fillna(fill_value, inplace=True)
        else:
            fill_value = df[col].mode()[0]
            df[col].fillna(fill_value, inplace=True)

    # 2. 数据增强（可选）
    if enhance_noise and n_noise > 0:
        df = add_noise_samples(df, n_noise=n_noise, id_col=id_col)

    # 3. 删除ID列
    if id_col in df.columns:
        df = df.drop(columns=[id_col])

    # 4. 正则化连续特征（排除标签列）
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) and col != label_col]
    df[numeric_cols] = df[numeric_cols].apply(lambda x: (x - x.mean()) / x.std(), axis=0)

    # 5. 对分类特征独热编码
    df = pd.get_dummies(df, dummy_na=True)

    return df

In [24]:
train_data = pd.read_csv('DataSet/train.csv')
train_data = preprocess_data(train_data, enhance_noise=True, n_noise=20)
train_data.head()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_10056\693886218.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(fill_value, inplace=True)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_10056\693886218.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, 

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,SaleType_Oth,SaleType_WD,SaleType_nan,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial,SaleCondition_nan
0,0.068604,-0.228458,-0.207192,0.646437,-0.512723,1.048352,0.874520,0.509587,0.574463,-0.295374,...,False,True,False,False,False,False,False,True,False,False
1,-0.876778,0.451906,-0.091583,-0.075621,2.190948,0.154824,-0.432599,-0.578139,1.173304,-0.295374,...,False,True,False,False,False,False,False,True,False,False
2,0.068604,-0.092386,0.074290,0.646437,-0.512723,0.982164,0.826108,0.320900,0.090106,-0.295374,...,False,True,False,False,False,False,False,True,False,False
3,0.304950,-0.455247,-0.096610,0.646437,-0.512723,-1.863885,-0.723069,-0.578139,-0.504332,-0.295374,...,False,True,False,True,False,False,False,False,False,False
4,0.068604,0.633337,0.376884,1.368496,-0.512723,0.949071,0.729284,1.364228,0.462180,-0.295374,...,False,True,False,False,False,False,False,True,False,False
